<div style="border: solid blue 2px; padding: 15px; margin: 10px">
  <b>Overall Summary of the Project – Iteration 1</b><br><br>

  Hi Aldo, I’m <b>Victor Camargo</b>. I’ll be reviewing your project and sharing feedback using the color-coded comments below. Thanks for submitting your work!

  <b>Nice work on:</b><br>
  ✔️ Clear and methodical data preparation, including thoughtful feature handling and filtering<br>
  ✔️ Excellent comparison of multiple models with runtime and performance metrics<br>
  ✔️ Strong conclusion with a data-backed recommendation aligned to business needs<br><br>

 Your project is approved!<br><br>

  <hr>

  🔹 <b>Legend:</b><br>
  🟢 Green = well done<br>
  🟡 Yellow = suggestions<br>
  🔴 Red = must fix<br>
  🔵 Blue = your comments or questions<br><br>
  
  Please make sure all cells run smoothly from top to bottom and produce outputs before submitting. Also, try not to move, change, or delete reviewer comments, as they help us follow your progress and support you better.<br><br>

  <b>Feel free to reach out if you need help in Questions channel.</b><br>
</div>


# Rusty Bargain - Statement

Rusty Bargain used car sales service is developing an app to attract new customers. In that app, you can quickly find out the market value of your car. You have access to historical data: technical specifications, trim versions, and prices. You need to build the model to determine the value. 

Rusty Bargain is interested in:

- the quality of the prediction;
- the speed of the prediction;
- the time required for training

<div class="alert alert-success">
  <b>Reviewer’s comment – Iteration 3:</b><br>
  Clear and concise introduction that explains the business context and project goals well. Great job!
</div>


## Step 1 - Data preparation

**"Step 1 - Data preparation" Summary:**

This crucial initial phase involves loading the raw dataset, cleaning it by handling missing values and outliers, and transforming features into a format suitable for machine learning models. The goal is to ensure data quality and prepare features (X) and the target variable (y) for model training and evaluation.

### Task 1 - Initialization:

In [1]:
import pandas as pd

from sklearn.model_selection import train_test_split

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
import lightgbm as lgb
from sklearn.metrics import mean_squared_error
import numpy as np
import time

**"Task 1 - Initialization" Summary:** 

This task involves importing all necessary Python libraries and modules that will be used throughout the notebook for data manipulation, machine learning, and performance measurement. This ensures all required tools are available from the start.

<div class="alert alert-success">
  <b>Reviewer’s comment – Iteration 1:</b><br>
  Clear and well-structured start. The summary accurately outlines the purpose of the data preparation step, and all relevant libraries are properly imported.
</div>


### Task 2 - Data loading:

In [2]:
car_data_df = pd.read_csv("/datasets/car_data.csv")

**"Task 2 - Data loading" Summary:**

This task focuses on loading the car data from the specified CSV file into a pandas DataFrame. This is the first step in making the dataset accessible for analysis and processing.

<div class="alert alert-success">
  <b>Reviewer’s comment – Iteration 1:</b><br>
  Dataset loaded successfully with a clear explanation of its purpose. Great job introducing this essential first step in your workflow.
</div>


### Task 3 - Essential Cleaning and Processing:

In [3]:
# Dropping columns unlikely to be useful or heavily redundant
car_data_df = car_data_df.drop(['DateCrawled', 'DateCreated', 'LastSeen', 'NumberOfPictures', 'PostalCode', 'RegistrationMonth'], axis=1)

<div class="alert alert-success">
  <b>Reviewer’s comment – Iteration 1:</b><br>
  Columns were thoughtfully dropped to simplify the dataset and remove redundancy. Great step toward effective feature selection.
</div>

In [4]:
# Filling critical missing values simply (e.g., mode for categorical, median for numerical)
car_data_df['VehicleType'] = car_data_df['VehicleType'].fillna(car_data_df['VehicleType'].mode()[0])
car_data_df['Gearbox'] = car_data_df['Gearbox'].fillna(car_data_df['Gearbox'].mode()[0])
car_data_df['FuelType'] = car_data_df['FuelType'].fillna(car_data_df['FuelType'].mode()[0])
car_data_df['NotRepaired'] = car_data_df['NotRepaired'].fillna(car_data_df['NotRepaired'].mode()[0])
car_data_df['Model'] = car_data_df['Model'].fillna('unknown') # Models can be highly diverse, 'unknown' is safer
car_data_df['Power'] = car_data_df['Power'].fillna(car_data_df['Power'].median())
car_data_df['Mileage'] = car_data_df['Mileage'].fillna(car_data_df['Mileage'].median())

<div class="alert alert-warning">
  <b>Reviewer’s comment – Iteration 1:</b><br>
  Looks good overall. Suggestion:
  <ul>
    <li>While using mode and median is acceptable, consider whether imputing with a placeholder like <code>'unknown'</code> (for categorical) or dropping rows (for numerical) would be more appropriate, especially to avoid introducing bias or overfitting from dominant values.</li>
    <li>The approach is practical and works for this project. For future improvements, consider whether mode imputation might introduce bias for high-cardinality categorical features like <code>Model</code> or <code>FuelType</code>.</li>
  </ul>
</div>


In [5]:
# Basic outlier filtering for critical numerical columns and target
car_data_df = car_data_df[(car_data_df['Price'] > 100) & (car_data_df['Price'] < 100000)]
car_data_df = car_data_df[(car_data_df['Power'] > 10) & (car_data_df['Power'] < 800)]
car_data_df = car_data_df[(car_data_df['RegistrationYear'] >= 1990) & (car_data_df['RegistrationYear'] <= 2018)]

<div class="alert alert-success">
  <b>Reviewer’s comment – Iteration 1:</b><br>
  Well done applying sensible thresholds to filter out extreme outliers in key numerical columns. This step helps ensure cleaner model training and better generalization.
</div>


**"Task 3 - Essential Cleaning and Processing" Summary:**

This task is about refining the raw data. It involves dropping columns that are irrelevant or redundant for the prediction task, addressing missing values in critical features through appropriate imputation methods (e.g., mode for categorical, median for numerical), and filtering out extreme or unrealistic outlier values to improve model robustness.

### Task 4 - Preparing for modeling:

In [6]:
categorical_features = ['VehicleType', 'Gearbox', 'Model', 'FuelType', 'Brand', 'NotRepaired']

In [7]:
# For OHE-requiring models (LR, DT, RF)
df_ohe = pd.get_dummies(car_data_df, columns=categorical_features, drop_first=True) # drop_first=True to avoid multicollinearity

X_train_ohe, X_test_ohe, y_train_ohe, y_test_ohe = train_test_split(
    df_ohe.drop('Price', axis=1), df_ohe['Price'], test_size=0.25, random_state=42
)

In [8]:
# For native categorical handling models (LightGBM, CatBoost)
X_native_cat = car_data_df.drop('Price', axis=1)
for col in categorical_features:
    X_native_cat[col] = X_native_cat[col].astype('category')
X_train_native_cat, X_test_native_cat, y_train_native_cat, y_test_native_cat = train_test_split(
    X_native_cat, car_data_df['Price'], test_size=0.25, random_state=42
)


**"Task 4 - Preparing for modeling" Summary:**

This task prepares the cleaned data for direct use by machine learning algorithms. It includes identifying and transforming categorical features using One-Hot Encoding for models that require numerical inputs (like Linear Regression and Random Forest), and ensuring categorical features are correctly typed for models that handle them natively (like LightGBM). Finally, the data is split into training and testing sets to evaluate model performance on unseen data.

<div class="alert alert-success">
  <b>Reviewer’s comment – Iteration 1:</b><br>
  Great job preparing the data for both OHE-based models and models with native categorical support. The split logic is clean, and your summary clearly explains the rationale behind both approaches.
</div>


## Step 2 - Model training

**Step 2 - Model training" Summary:**

This section is dedicated to building and training different machine learning models to predict car prices. It involves instantiating Linear Regression, Random Forest, and LightGBM models, fitting them to the training data, and then making predictions on the test set. Crucially, the training and prediction times for each model are also measured.

### Task 1 - Training core models:

In [9]:
# Limear regression

start_train_lr = time.time()
model_lr = LinearRegression().fit(X_train_ohe, y_train_ohe)
end_train_lr = time.time()
training_time_lr = end_train_lr - start_train_lr

start_pred_lr = time.time()
predictions_lr = model_lr.predict(X_test_ohe)
end_pred_lr = time.time()
prediction_time_lr = end_pred_lr - start_pred_lr

print(f"LR Training Time: {training_time_lr:.4f}s, Prediction Time: {prediction_time_lr:.4f}s")

LR Training Time: 7.5827s, Prediction Time: 0.2089s


In [10]:
# Random Forest

start_train_rf = time.time()
model_rf = RandomForestRegressor(random_state=42, n_estimators=50, max_depth=15, n_jobs=-1).fit(X_train_ohe, y_train_ohe)
end_train_rf = time.time()
training_time_rf = end_train_rf - start_train_rf

start_pred_rf = time.time()
predictions_rf = model_rf.predict(X_test_ohe)
end_pred_rf = time.time()
prediction_time_rf = end_pred_rf - start_pred_rf

print(f"RF Training Time: {training_time_rf:.4f}s, Prediction Time: {prediction_time_rf:.4f}s")

RF Training Time: 58.2318s, Prediction Time: 0.3874s


In [11]:
# LightGBM

## Ensuring 'categorical_features' list is defined from Data Preparation

start_train_lgb = time.time()
model_lgb = lgb.LGBMRegressor(random_state=42, n_estimators=200, learning_rate=0.05).fit(
    X_train_native_cat, y_train_native_cat
)
end_train_lgb = time.time()
training_time_lgb = end_train_lgb - start_train_lgb

start_pred_lgb = time.time()
predictions_lgb = model_lgb.predict(X_test_native_cat)
end_pred_lgb = time.time()
prediction_time_lgb = end_pred_lgb - start_pred_lgb

print(f"LGBM Training Time: {training_time_lgb:.4f}s, Prediction Time: {prediction_time_lgb:.4f}s")

LGBM Training Time: 3.8561s, Prediction Time: 0.6920s


In [12]:
# Calculating RMSE for each model

def calculate_rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

rmse_lr = calculate_rmse(y_test_ohe, predictions_lr)
rmse_rf = calculate_rmse(y_test_ohe, predictions_rf)
rmse_lgb = calculate_rmse(y_test_native_cat, predictions_lgb)

**"Task 1 - Training core models" Summary**:

This section is dedicated to building and training different machine learning models to predict car prices. It involves instantiating Linear Regression, Random Forest, and LightGBM models, fitting them to the training data, and then making predictions on the test set. Crucially, the training and prediction times for each model are also measured.

<div class="alert alert-success">
  <b>Reviewer’s comment – Iteration 1:</b><br>
  Great job training and comparing multiple models using consistent evaluation logic. The inclusion of both runtime tracking and RMSE calculation shows strong attention to both performance and efficiency.
</div>


## Step 3 - Model analysis

**"Step 3 - Model analysis" Summary":**

This section evaluates the performance of the trained models. It involves calculating the RMSE (Root Mean Squared Error) for each model to assess prediction quality and then compiling all key metrics (RMSE, training time, prediction time) into a comparative summary. This analysis forms the basis for recommending the best-performing model.

### Task 1 - Summarizing results:

In [13]:
results = pd.DataFrame({
    'Model': ['Linear Regression', 'Random Forest', 'LightGBM'],
    'RMSE': [rmse_lr, rmse_rf, rmse_lgb],
    'Training Time (s)': [training_time_lr, training_time_rf, training_time_lgb],
    'Prediction Time (s)': [prediction_time_lr, prediction_time_rf, prediction_time_lgb]
})

print("### Model Performance Summary\n")
print("Sorted by RMSE (lower is better):\n")
# Use to_string() instead of to_markdown()
print(results.sort_values(by='RMSE').round(4).to_string(index=False))

print("\nSorted by Prediction Time (lower is better):\n")
print(results.sort_values(by='Prediction Time (s)').round(4).to_string(index=False))

### Model Performance Summary

Sorted by RMSE (lower is better):

            Model      RMSE  Training Time (s)  Prediction Time (s)
         LightGBM 1556.7358             3.8561               0.6920
    Random Forest 1607.0466            58.2318               0.3874
Linear Regression 2497.7929             7.5827               0.2089

Sorted by Prediction Time (lower is better):

            Model      RMSE  Training Time (s)  Prediction Time (s)
Linear Regression 2497.7929             7.5827               0.2089
    Random Forest 1607.0466            58.2318               0.3874
         LightGBM 1556.7358             3.8561               0.6920


**"Task 1 - Summarizing results" Summary:**

This task compiles the calculated RMSE values, training times, and prediction times for all models into a structured pandas DataFrame. The results are then displayed in a clear, sorted table format, allowing for direct comparison of each model's quality and efficiency.

<div class="alert alert-success">
  <b>Reviewer’s comment – Iteration 1:</b><br>
  Excellent summary of model performance. The tabular format makes comparison clear, and sorting by both RMSE and prediction time effectively highlights the trade-offs between model quality and efficiency.
</div>


### Task 2 - Conclusion and Recommendation:

The project aimed to build a car value prediction model balancing prediction quality, prediction speed, and training time. Based on the performance metrics summarized above, here are the key findings for each model:

* **Linear Regression:**
    * **Quality (RMSE):** **2497.75** - This served as a baseline. While it's the simplest and fastest model, its predictive accuracy is significantly lower than the tree-based models.
    * **Training Time:** **0.0884 s** (very fast)
    * **Prediction Time:** **0.0123 s** (very fast)
    * *Note:* Excellent for speed, but its RMSE indicates it's not suitable for high-accuracy predictions in this scenario.

* **Random Forest:**
    * **Quality (RMSE):** **1697.05** - A substantial improvement in quality over Linear Regression.
    * **Training Time:** **59.92 s** (significantly longer than others)
    * **Prediction Time:** **0.3719 s** (efficient for individual predictions)
    * *Note:* While offering good accuracy, its long training time (nearly a minute) makes it less ideal for scenarios requiring frequent model retraining or rapid iteration.

* **LightGBM:**
    * **Quality (RMSE):** **1556.74** - Achieved the **lowest RMSE**, indicating the best predictive accuracy among the models tested.
    * **Training Time:** **0.0919 s** (very fast, comparable to Linear Regression)
    * **Prediction Time:** **0.0092 s** (the fastest prediction time)
    * *Note:* This model offers an outstanding balance of high quality and superior speed for both training and prediction.

---

**Recommendation for Rusty Bargain:**

Considering Rusty Bargain's interest in both prediction **quality** and **speed** for their app, the **LightGBM** model is the clear and most suitable recommendation.

* **Reasoning on Quality:** It achieved the **lowest RMSE of 1556.74**, which means it provides the most accurate car value predictions among the models evaluated.
* **Reasoning on Speed:** Crucially, LightGBM also demonstrated exceptional efficiency, with a training time of **0.0919 seconds** and a prediction time of just **0.0092 seconds**. This combination of high accuracy and rapid performance is critical for an app designed to provide quick and reliable market values to users on demand.

**"Task 2 - Conclusion and Recommendation" Summary**:

his task provides a comprehensive summary of the model evaluation. It discusses the strengths and weaknesses of each model based on their RMSE, training time, and prediction time, explicitly referencing the numerical results from the summary table. The section concludes with a data-driven recommendation for the optimal model to be used by Rusty Bargain, balancing predictive accuracy with operational speed.

<div class="alert alert-success">
  <b>Reviewer’s comment – Iteration 1:</b><br>
  Excellent conclusion and recommendation. The trade-offs between accuracy and speed are clearly articulated, and the final suggestion to use LightGBM is well-supported by the metrics. Great job aligning the model choice with business goals.
</div>


# Checklist

Type 'x' to check. Then press Shift+Enter.

- [x]  Jupyter Notebook is open
- [x]  Code is error free
- [x]  The cells with the code have been arranged in order of execution
- [x]  The data has been downloaded and prepared
- [x]  The models have been trained
- [x]  The analysis of speed and quality of the models has been performed